<a href="https://colab.research.google.com/github/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/blob/main/Notebooks/M02_04_Agregacion_exportacion_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
  <img src="https://github.com/paulmunozpauta/Curso_Introduccion_Ciencia_Datos/raw/main/Static/Imgs/UdeC_color_horizontal.jpg" width="500">
</p>

<p align="center"><b style="font-size:28px;">Facultad de Ingeniería Agrícola</b></p>
<p align="center"><b style="font-size:28px;">Introducción a la Ciencia de Datos</b></p>
<hr>
<p align="center"><b style="font-size:28px;">Contacto</b></p>
<p align="center">
  paulmunoz@udec.cl<br>
  https://paulmunoz.com
</p>

# Introducción a la Ciencia de Datos

## Módulo 2: Organización y preparación de datos

### Notebook 4: Agregación temporal y exportación de datos

En los notebooks anteriores aprendimos a importar, explorar, organizar, seleccionar y limpiar datos. También trabajamos con fechas, índices temporales y control de calidad.

En este notebook aprenderemos a trabajar con datos registrados a diferentes **resoluciones temporales**. Partiremos de observaciones meteorológicas registradas cada **15 minutos** y las transformaremos a escalas horarias, diarias y mensuales.

Este proceso se denomina **agregación temporal** y es una operación frecuente cuando trabajamos con datos ambientales obtenidos mediante sensores automáticos.

También aprenderemos a exportar los datos procesados para utilizarlos posteriormente en otros análisis.

### Objetivos

Al finalizar este notebook podrás:

- identificar la resolución temporal de una serie;
- comprender el concepto de agregación temporal;
- utilizar `resample()` para cambiar la resolución temporal de los datos;
- transformar datos de 15 minutos a escalas horarias, diarias y mensuales;
- aplicar funciones de agregación como `sum()`, `mean()`, `min()` y `max()`;
- seleccionar la función de agregación apropiada según la variable analizada;
- exportar los datos procesados a archivos CSV y Excel.

## Datos utilizados: estación meteorológica CH01 – Campus Chillán UdeC

En este notebook trabajaremos con datos provenientes de la estación meteorológica **CH01**, ubicada en el Campus Chillán de la Universidad de Concepción.

La estación forma parte de un sistema de monitoreo meteorológico de bajo costo que registra automáticamente diferentes variables ambientales.

Entre las variables monitoreadas se encuentran:

- temperatura del aire;
- humedad relativa;
- presión atmosférica;
- precipitación;
- velocidad y dirección del viento;
- radiación solar;
- índice UV.

Los datos de la estación pueden visualizarse en tiempo real en la plataforma **MeteoChillán UdeC**.

**Red de estaciones:**  
https://meteochillan.udec.cl/stations.html

**Estación CH01 – Campus Chillán:**  
https://meteochillan.udec.cl/stations/ch_01.html

Para este ejercicio utilizaremos registros históricos con una **resolución temporal de 15 minutos**.

Trabajar con datos de alta frecuencia nos permitirá estudiar cómo una misma serie puede representarse a diferentes escalas temporales:

**15 minutos → horario → diario → mensual**

## 1. Resolución temporal de los datos

La **resolución temporal** indica el intervalo de tiempo entre observaciones consecutivas de una serie.

Por ejemplo, un sensor que registra información cada 15 minutos genera observaciones como:

| Fecha y hora | Temperatura (°C) |
|---|---:|
| 2026-08-10 10:00 | 8.2 |
| 2026-08-10 10:15 | 8.6 |
| 2026-08-10 10:30 | 9.1 |
| 2026-08-10 10:45 | 9.4 |
| 2026-08-10 11:00 | 9.8 |

En este caso, la resolución temporal de los datos es de **15 minutos**.

Una alta resolución temporal permite observar cambios que ocurren dentro de un mismo día. Sin embargo, dependiendo de la pregunta que queremos responder, puede ser más conveniente trabajar con información horaria, diaria o mensual.

Por ejemplo:

- **15 minutos:** ¿cómo cambia la temperatura durante la mañana?
- **horaria:** ¿cuál fue la temperatura media de cada hora?
- **diaria:** ¿cuál fue la temperatura media, mínima o máxima de cada día?
- **mensual:** ¿cómo cambió la temperatura media entre diferentes meses?

La resolución temporal utilizada depende, por lo tanto, del objetivo del análisis.

## 2. Agregación temporal

La **agregación temporal** consiste en agrupar varias observaciones correspondientes a un determinado intervalo de tiempo y resumirlas mediante una operación.

Por ejemplo, nuestra estación registra cuatro valores de temperatura durante una hora:

| Fecha y hora | Temperatura (°C) |
|---|---:|
| 10:00 | 8.2 |
| 10:15 | 8.6 |
| 10:30 | 9.1 |
| 10:45 | 9.4 |

Estas cuatro observaciones pueden utilizarse para obtener un único valor que represente esa hora.

Por ejemplo, podemos calcular la **temperatura media horaria**.

De esta manera:

**4 observaciones de 15 minutos → 1 observación horaria**

El mismo procedimiento puede aplicarse posteriormente para obtener datos diarios o mensuales:

**15 minutos → horario → diario → mensual**

Al agregar los datos disminuye el número de observaciones y cambia la información representada por cada valor.

## 3. La operación depende de la variable

Agregar datos no significa simplemente cambiar su frecuencia temporal. También debemos decidir **qué operación utilizar para resumir las observaciones**.

La operación apropiada depende de la variable y de la pregunta que queremos responder.

### Temperatura

A partir de observaciones de temperatura cada 15 minutos podemos calcular, por ejemplo:

- `mean()` → temperatura media del periodo;
- `min()` → temperatura mínima del periodo;
- `max()` → temperatura máxima del periodo.

### Precipitación

Si los registros representan la precipitación acumulada durante cada intervalo, podemos utilizar:

- `sum()` → precipitación total acumulada durante el periodo.

Por ejemplo:

**Temperatura cada 15 min → `mean()` → temperatura media horaria**

**Precipitación cada 15 min → `sum()` → precipitación acumulada horaria**

Por lo tanto, antes de realizar una agregación debemos preguntarnos:

1. ¿Qué representa cada observación original?
2. ¿Qué periodo queremos representar?
3. ¿Qué operación tiene sentido para esa variable?

Python puede ejecutar diferentes operaciones matemáticas, pero **no todas tienen necesariamente una interpretación adecuada para la variable analizada**.

## 4. Cargar y explorar los datos de la estación CH01

Ahora trabajaremos con los registros reales de la estación CH01.

Antes de realizar cualquier agregación debemos conocer la estructura del archivo y comprobar:

- qué columnas contiene;
- cómo están almacenadas la fecha y la hora;
- qué variables meteorológicas están disponibles;
- qué unidades utilizan;
- cuántos registros contiene la serie;
- cuál es el periodo disponible.

Primero importaremos las librerías necesarias.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt